# Dispatch and unit-commitment demo

Run from the repo root (open the folder in VS Code, then Run All). The fleet is a stylized, invented 8-unit system; see the README for assumptions.

In [ ]:
import sys; sys.path.insert(0, '..')   # lets the notebook import src/ from the repo root
import pandas as pd
from src import analysis as an, model
from src.data import load_inputs, marginal_costs, merit_order, BASE_GAS_PRICE

inputs = load_inputs('../data')
gas = BASE_GAS_PRICE
pd.Series(marginal_costs(inputs, gas)).round(2).sort_values()

## Phase 1: economic dispatch (LP)

In [ ]:
ed = model.run_ed(inputs, gas)
print(ed.info)
print('merit-order violations:', an.merit_order_violations(ed.dispatch, inputs, gas))
ed.dispatch.round(1)

In [ ]:
prices = an.ed_marginal_prices(inputs, gas)
pd.DataFrame({'price': prices.round(2), 'marginal_unit': an.marginal_unit_ed(ed.dispatch, inputs, gas)}).T

## Phase 2: unit commitment (MILP)

In [ ]:
uc = model.run_uc(inputs, gas)
print(uc.info)
an.verify_uc_solution(uc.dispatch, uc.commitment, inputs)

In [ ]:
print('UC cost - ED cost: $', round(uc.info.objective - ed.info.objective))
uc.commitment.T

## Charts

In [ ]:
an.plot_dispatch_stack(uc.dispatch, inputs, '../docs/uc_dispatch.png', title='Unit commitment dispatch')
an.plot_commitment_heatmap(uc.commitment, '../docs/commitment_heatmap.png')
from IPython.display import Image, display
display(Image('../docs/uc_dispatch.png')); display(Image('../docs/commitment_heatmap.png'))

## Phase 3

The full sensitivity analysis (gas sweep, break-even check, wind and reserve scenarios) is in `run_all.py`; run it from the repo root with `python run_all.py`.